# CosMx pathway enrichment analysis — epithelial cells (t-stat ranked GSEA)

**Goal:** Identify pathways enriched in MHC II positive vs negative malignant cells
using GSEA on a pre-ranked gene list derived from t-test differential expression
scores. This approach is more sensitive than per-cell scoring as it directly uses
the differential expression signal between groups. Results are compared to the
scRNA-seq pathway analysis to identify concordantly enriched pathways.

**Input:** `outputs/processed/epithelial.h5ad` — CosMx epithelial cells with
patient classification; `data/cosmx_pathway_annotations.csv`.

**Output:** `outputs/tables/figure3/cosmx_gsea_hallmark_t_test.csv`;
`outputs/tables/figure3/cosmx_gsea_cosmx_pathways_t_test.csv`

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from scipy import stats
from statsmodels.stats.multitest import multipletests

import anndata as ad
import scanpy as sc
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
epithelial_path      = repo_root / cfg['outputs']['processed'] / 'epithelial.h5ad'
pathway_annot_path   = repo_root / cfg['datasets']['cosmx']['pathway_annotations']
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])

# outputs
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

CosMx epithelial cells are loaded and subset to MHC II positive and negative
patients only. Clonal and no-malignant-cells groups are excluded from this
comparison. Gene expression is already normalized and log1p-transformed from
the upstream `mhc2_ihc_deg` notebook.

In [ ]:
# load epithelial cells
epithelial = ad.read_h5ad(epithelial_path)
print(f"Epithelial cells: {epithelial.n_obs:,} × {epithelial.n_vars} genes")

# load and merge patient classifications — not stored in epithelial.h5ad
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index.astype(str)

epithelial.obs['patient'] = epithelial.obs['patient'].astype(str)
epithelial.obs = pd.merge(
    epithelial.obs, patient_classifications, on='patient', how='left'
)

print(epithelial.obs['patient classification'].value_counts())

# subset to MHC II pos and neg only
high_low = epithelial[
    epithelial.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()
print(f"\nPos + neg only: {high_low.n_obs:,} cells")

## Load gene sets

Two gene set collections are used:

1. **NanoString CosMx pathway annotations** — gene sets curated specifically for
the CosMx 1000-plex panel, covering oncogenic signaling, immune, and metabolic
pathways. These provide a targeted view using only genes present in the panel.

2. **MSigDB Hallmark gene sets** — 50 well-curated, biologically coherent gene
sets representing well-defined biological states. Used as the primary figure panel
given their broad recognition in the field.

In [ ]:
# load MSigDB Hallmark gene sets via decoupler
hallmark = dc.op.hallmark(organism='human', license='academic', verbose=False)
hallmark = hallmark[~hallmark.duplicated(['source', 'target'])]
print(f"Hallmark gene sets: {hallmark['source'].nunique()}")

In [ ]:
# load NanoString CosMx pathway annotations
pathways = pd.read_csv(pathway_annot_path, index_col=0)

# reformat to long format for decoupler
long_df = (
    pathways.reset_index()
    .rename(columns={'Gene': 'genesymbol'})
    .melt(id_vars='genesymbol', var_name='geneset', value_name='Belongs')
    .query('Belongs == 1')
    .drop(columns='Belongs')
    .drop_duplicates(subset=['genesymbol', 'geneset'])
    .reset_index(drop=True)
)
long_df['collection'] = 'CosMx'
long_df['geneset'] = 'CosMx: ' + long_df['geneset'].astype(str)
print(f"CosMx pathway sets: {long_df['geneset'].nunique()}")


## Rank genes by differential expression

Genes are ranked by t-test scores comparing class II positive vs negative
epithelial cells. The full 976-gene CosMx panel is used without HVG filtering —
the panel is already a curated gene set and filtering further reduces statistical
power. Scores are rank-transformed before GSEA for numerical stability.

In [ ]:
import warnings

# rank genes by t-test between class II positive and negative
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(
        high_low,
        groupby='patient classification',
        method='t-test',
        key_added='t-test',
        use_raw=False,
    )

# use full gene ranking — CosMx panel is already a curated 976-gene set
# HVG filter not applied as it reduces power on this sparse panel
t_stats = (
    sc.get.rank_genes_groups_df(high_low, 'class II positive', key='t-test')
    .set_index('names')
    .sort_values('scores', key=np.abs, ascending=False)[['scores']]
)

t_stats['scores'] = (
    t_stats['scores'].rank(method='average')
    - t_stats['scores'].rank(method='average').mean()
)
t_stats['scores'] /= t_stats['scores'].std()

print(f"Genes ranked: {len(t_stats)}")
print(t_stats.head())

## GSEA helper function

`run_gsea_tstat` runs GSEA on a pre-ranked gene list rather than per-cell
scores. The t-stats DataFrame is transposed before passing to `dc.mt.gsea`
so genes are columns and the single ranking is the one row. Results are
returned directly as a DataFrame.

The same function is used in `scrna_pathway_analysis_tstat.ipynb` to enable
direct comparison between modalities.

In [ ]:
def run_gsea_tstat(t_stats_df, net_df, collection_name, times=200, min_genes_in_set=5):
    """
    Run GSEA on a pre-ranked t-stat gene list.

    Parameters
    ----------
    t_stats_df : pd.DataFrame
        Pre-ranked gene list with 'scores' column, indexed by gene symbol.
    net_df : pd.DataFrame
        Gene sets in long format with columns ['source'/'geneset', 'target'/'genesymbol'].
    collection_name : str
        Label for this collection (used in progress messages).
    times : int
        Number of permutations for significance testing.
    min_genes_in_set : int
        Minimum number of genes required per gene set.

    Returns
    -------
    pd.DataFrame
        GSEA results with ES, NES, pval, padj columns.
    """

    if 'geneset' in net_df.columns:
        net = net_df.rename(columns={'geneset': 'source', 'genesymbol': 'target'})
    else:
        net = net_df.copy()

    net = net[~net.duplicated(['source', 'target'])]
    valid_sets = (
        net.groupby('source')['target']
        .nunique()
        .loc[lambda x: x >= min_genes_in_set]
        .index
    )
    net = net[net['source'].isin(valid_sets)]
    print(f"Running t-stat GSEA for {collection_name}: {net['source'].nunique()} gene sets")

    scores, pvals = dc.mt.gsea(
        t_stats_df.T,
        net=net,
        tmin=min_genes_in_set,
        times=times,
        seed=1,
        verbose=False,
    )

    # assemble results DataFrame
    results = pd.DataFrame({
        'NES': scores.iloc[0],
        'pval': pvals.iloc[0],
    })
    results['padj'] = multipletests(results['pval'], method='fdr_bh')[1]
    results['-log10_padj'] = -np.log10(results['padj'].replace(0, np.nextafter(0, 1)))
    results = results.sort_values('padj')

    return results

## Run GSEA

GSEA is run for Hallmark and CosMx pathway collections on the pre-ranked
t-stat gene list. Results are saved as checkpoints.

In [ ]:
%%time

hallmark_results_path = table_out / 'cosmx_gsea_hallmark_t_test.csv'
cosmx_results_path    = table_out / 'cosmx_gsea_cosmx_pathways_t_test.csv'

# --- Hallmark ---
if hallmark_results_path.exists():
    print("Loading pre-computed Hallmark t-stat GSEA results...")
    gsea_hallmark = pd.read_csv(hallmark_results_path, index_col=0)
else:
    print("Running Hallmark t-stat GSEA...")
    gsea_hallmark = run_gsea_tstat(
        t_stats, hallmark, 'hallmark', times=1000, min_genes_in_set=5
    )
    gsea_hallmark.to_csv(hallmark_results_path)
    print(f"Saved → {hallmark_results_path}")

# --- CosMx pathways ---
if cosmx_results_path.exists():
    print("Loading pre-computed CosMx t-stat GSEA results...")
    gsea_cosmx = pd.read_csv(cosmx_results_path, index_col=0)
else:
    print("Running CosMx t-stat GSEA...")
    gsea_cosmx = run_gsea_tstat(
        t_stats, long_df, 'CosMx', times=1000, min_genes_in_set=5
    )
    gsea_cosmx.to_csv(cosmx_results_path)
    print(f"Saved → {cosmx_results_path}")

print("\nTop Hallmark pathways (t-stat):")
print(gsea_hallmark.sort_values('padj').head(10))

## Visualization

Pathway enrichment results are visualized as a barplot ranked by absolute NES
and a volcano plot of NES vs -log10(FDR). Both plots use the paper color palette
— orange for MHC II high/positive, purple for MHC II low/negative.

In [ ]:
def plot_gsea_barplot(gsea_df, top_n=25, title=None, figsize=(10, 10)):
    plot_df = (
        gsea_df.copy()
        .assign(Direction=np.where(gsea_df['NES'] >= 0, 'MHC-II High', 'MHC-II Low'))
        .assign(absNES=lambda d: d['NES'].abs())
        .sort_values('absNES', ascending=False)
        .head(top_n)
        .sort_values('NES')
    )
    # clean up pathway names
    plot_df.index = plot_df.index.str.replace('_', ' ').str.title()

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=plot_df, y=plot_df.index, x='NES',
        hue='Direction', dodge=False,
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', ax=ax,
    )
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Normalized Enrichment Score (NES)')
    ax.set_ylabel('')
    ax.set_title(title or f'Top {top_n} Pathways by |NES|')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig


def plot_gsea_volcano(gsea_df, sig_thresh=0.05, nes_thresh=None,
                      top_n_labels=15, title=None, figsize=(10, 8), ymax=30):
    
    score_col = 'NES' if 'NES' in gsea_df.columns else 'log2fc'
    
    # auto-set threshold based on score type
    if nes_thresh is None:
        nes_thresh = 1.5 if score_col == 'NES' else gsea_df[score_col].abs().quantile(0.75)
    
    # rest of function unchanged...
    df = gsea_df.copy()
    df['-log10_FDR'] = -np.log10(df['padj'].replace(0, np.nextafter(0, 1)))
    df['-log10_FDR'] = df['-log10_FDR'].clip(upper=ymax)  # cap floored values
    df['Direction'] = np.where(df['NES'] >= 0, 'MHC-II High', 'MHC-II Low')

    fig, ax = plt.subplots(figsize=figsize)
    sns.scatterplot(
        data=df, x='NES', y='-log10_FDR', hue='Direction',
        palette={'MHC-II High': cmap_high_low[0], 'MHC-II Low': cmap_high_low[1]},
        edgecolor='none', alpha=0.9, ax=ax,
    )
    ax.axvline( nes_thresh, color='black', lw=1, ls='--')
    ax.axvline(-nes_thresh, color='black', lw=1, ls='--')
    ax.axhline(-np.log10(sig_thresh), color='black', lw=1, ls='--')

    # note capped values
    ax.text(0.02, 0.98, f'Note: -log10(FDR) capped at {ymax} for display',
            transform=ax.transAxes, fontsize=8, va='top', color='gray')

    top_hits = df.sort_values('padj').head(top_n_labels)
    texts = []
    for _, row in top_hits.iterrows():
        texts.append(ax.text(
            row['NES'], row['-log10_FDR'],
            row.name.replace('_', ' ').title(),
            fontsize=8,
            ha='right' if row['NES'] < 0 else 'left',
        ))
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5), ax=ax)

    ax.set_xlabel('Normalized Enrichment Score (NES)')
    ax.set_ylabel(r'$-\log_{10}$(FDR)')
    ax.set_title(title or 'GSEA Volcano')
    ax.legend(title='Enriched In', bbox_to_anchor=(1.05, 1), loc='upper left')
    sns.despine()
    plt.tight_layout()
    return fig

In [ ]:
# Hallmark barplot
fig = plot_gsea_barplot(
    gsea_hallmark, top_n=25,
    title='Hallmark Pathways — MHC II pos vs neg (CosMx epithelial)',
)
fig.savefig(supp_out / 'figS3_cosmx_hallmark_gsea_barplot_ttest.pdf', bbox_inches='tight')
plt.show()

# Hallmark volcano
fig = plot_gsea_volcano(
    gsea_hallmark,
    title='Hallmark GSEA Volcano — MHC II pos vs neg (CosMx epithelial)',
)
fig.savefig(supp_out / 'figS3_cosmx_hallmark_gsea_volcano_ttest.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways barplot
fig = plot_gsea_barplot(
    gsea_cosmx, top_n=25,
    title='CosMx Pathways — MHC II pos vs neg (CosMx epithelial)',
)
fig.savefig(supp_out / 'figS3_cosmx_cosmx_pathways_gsea_barplot_ttest.pdf', bbox_inches='tight')
plt.show()

# CosMx pathways volcano
fig = plot_gsea_volcano(
    gsea_cosmx,
    title='CosMx Pathways Volcano — MHC II pos vs neg (CosMx epithelial)',
)
fig.savefig(supp_out / 'figS3_cosmx_cosmx_pathways_gsea_volcano_ttest.pdf', bbox_inches='tight')
plt.show()